# SPIDER ML Task 2 _ Base ML Level 3
## Transformer for Weather Forecasting

In this notebook, I will build a Transformer-based model for weather forecasting.The model will use past weather observations to predict future temperature values.

## Problem Statement

The goal is to build an encoder-only Transformer for time-series forecasting.We will use the Jena Climate dataset, prepare hourly data, create sequence windows, train the model and compare it with the custom LSTM from Level 2.

In [15]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Load Dataset

We load the Jena Climate dataset and prepare it for hourly forecasting.The original data is recorded every 10 minutes, so we will downsample it by averaging every 6 rows.

In [17]:
df = pd.read_csv("jena_climate_2009_2016.csv.zip")
df.head()

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


## Clean and Prepare Data

We convert the date column,sort the data and keep the useful weather features

In [18]:
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.sort_values("Date Time").reset_index(drop=True)

## Downsample to Hourly Data

We average every 6 rows to convert 10-minute readings into hourly readings

In [19]:
df_hourly = df.groupby(np.arange(len(df)) // 6).mean(numeric_only=True)
df_hourly.head()

,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,996.523333,-8.261667,265.161667,-9.063333,93.883333,3.271667,3.071667,0.200000,1.918333,3.081667,1308.973333,0.468333,0.940000,177.500000
1,996.545000,-8.203333,265.221667,-9.026667,93.733333,3.288333,3.081667,0.205000,1.926667,3.093333,1308.713333,0.323333,0.711667,172.016667
2,996.781667,-8.751667,264.653333,-9.591667,93.583333,3.146667,2.945000,0.200000,1.840000,2.955000,1311.805000,0.236667,0.606667,192.966667
3,997.011667,-8.936667,264.450000,-9.846667,93.050000,3.101667,2.885000,0.215000,1.803333,2.891667,1313.051667,0.163333,0.565000,169.216667
4,997.236667,-9.445000,263.923333,-10.450000,92.316667,2.980000,2.751667,0.231667,1.718333,2.756667,1315.951667,0.340000,0.753333,136.260000


## Select Features

We keep a small set of weather features for forecasting

In [20]:
features = ["T (degC)", "p (mbar)", "rho (g/m**3)", "wv (m/s)", "wd (deg)"]
data = df_hourly[features].copy()
data = data.ffill().bfill()

## Normalize Data

We scale the features for stable training

In [21]:
data_mean = data.mean()
data_std = data.std()
data_scaled = (data - data_mean) / data_std

## Create Sequences

using the previous 720 hours of weather data to predict the next 24 hours of temperature

In [22]:
def create_sequences(data_array, input_len=720, output_len=24):
    X, y = [], []
    for i in range(len(data_array) - input_len - output_len):
        X.append(data_array[i:i+input_len])
        y.append(data_array[i+input_len:i+input_len+output_len, 0])
    return np.array(X), np.array(y)

data_array = data_scaled.values
X, y = create_sequences(data_array, input_len=720, output_len=24)

print(X.shape, y.shape)

(69332, 720, 5) (69332, 24)


## Train-Test Split

We split the time series into training and test sets without shuffling

In [23]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(55465, 720, 5) (13867, 720, 5) (55465, 24) (13867, 24)


## Convert to Tensors

We convert the data into PyTorch tensors for training

In [24]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

## Create DataLoaders

We prepare mini-batches for efficient training

In [25]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Sinusoidal Positional Encoding

We implement positional encoding manually so the Transformer knows the order of the time steps

In [26]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## Transformer Model

We now build an encoder-only Transformer for temperature forecasting.The model uses the positional encoding, Transformer encoder layers, and a final linear layer to predict the next 24 hours

In [27]:
class TransformerForecast(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, output_size=24, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, output_size)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x[:, -1, :]
        out = self.fc(x)
        return out

## Model Setup

We define the model, loss function and optimizer

In [28]:
input_size = X_train.shape[2]

model = TransformerForecast(
    input_size=input_size,
    d_model=64,
    nhead=4,
    num_layers=2,
    output_size=24,
    dim_feedforward=128,
    dropout=0.1
).to(device)

criterion = nn.HuberLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Training Loop

We train the Transformer and track the losses

In [ ]:
train_losses = []
test_losses = []

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    test_running_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            test_running_loss += loss.item()

    test_loss = test_running_loss / len(test_loader)
    test_losses.append(test_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

## Evaluation

We now compute Huber Loss, MAE and MSE on the test set

In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y_batch.numpy())

all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

huber = np.mean(np.where(
    np.abs(all_preds - all_targets) < 1,
    0.5 * (all_preds - all_targets) ** 2,
    np.abs(all_preds - all_targets) - 0.5
))
mae = mean_absolute_error(all_targets, all_preds)
mse = mean_squared_error(all_targets, all_preds)

print("Huber Loss:", huber)
print("MAE:", mae)
print("MSE:", mse)

## Prediction Plot

We plot predicted temperature against actual temperature for one test window

In [ ]:
true_vals = all_targets[0]
pred_vals = all_preds[0]

plt.figure(figsize=(10, 5))
plt.plot(true_vals, label="Actual Temperature", marker="o")
plt.plot(pred_vals, label="Predicted Temperature", marker="x")
plt.xlabel("Forecast Hour")
plt.ylabel("Normalized Temperature")
plt.title("Predicted vs Actual Temperature")
plt.legend()
plt.grid(True)
plt.show()

## Save Model Weights

We save the trained model for submission

In [ ]:
torch.save(model.state_dict(), "transformer_level3.pth")
print("Model saved as transformer_level3.pth")

## Comparison With LSTM

We compare the Transformer against the custom LSTM from Level 2 using the same test split

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Transformer"],
    "Huber Loss": [huber],
    "MAE": [mae],
    "MSE": [mse]
})

comparison

## Conclusion

In this notebook, I implemented an encoder only Transformer for weather forecasting.The model uses manual sinusoidal positional encoding and is evaluated on hourly Jena Climate data